本章我们来采用策略迭代与价值迭代来求解悬崖漫步（Cliff Walking）这个环境中的最优策略

它要求一个智能体从起点出发，避开悬崖行走，最终到达目标位置；有一个 4×12 的网格世界，每一个网格表示一个状态。智能体的起点是左下角的状态，目标是右下角的状态，智能体在每一个状态都可以采取 4 种动作：上、下、左、右。如果智能体采取动作后触碰到边界墙壁则状态不发生改变，否则就会相应到达下一个状态。环境中有一段悬崖，智能体掉入悬崖或到达目标状态都会结束动作并回到起点，也就是说掉入悬崖或者达到目标状态是终止状态。智能体每走一步的奖励是 −1，掉入悬崖的奖励是 −100。

In [1]:
import copy

class CliffWalkingEnv:
    """
        悬崖漫步环境
    """
    def __init__(self, ncol=12, nrow=4):
        """
            初始化环境
            ncol: 列数
            nrow: 行数
        """
        self.ncol = ncol  # 定义网格世界的列
        self.nrow = nrow  # 定义网格世界的行

        # 转移矩阵P(s,a) = [(p, next_state, reward, done)]包含下一个状态和奖励
        self.P = self.createP()

    def createP(self):
        """
            创建转移矩阵P(s,a) = [(p, next_state, reward, done)]
            先来划定危险和结束区域：如果下一步的 y 是 3（最后一行），并且 x 大于 0（排除了第 0 列）。只要进了这个区域，游戏就直接结束
            界定悬崖位置和终点：如果下一列不是最后一列（即不是第11列），那么奖励就是 -100；如果下一列是最后一列（即第11列），奖励就是 0。
        """
        # 外层循环: i 遍历 48 个状态 (4*12)。内层循环: 给每个状态分配一个包含 4 个空列表的列表 (对应4个动作)
        # 最终 P 是一个拥有 48 个元素的列表，每个元素里又包含 4 个子列表
        P = [[[] for j in range(4)] for i in range(self.nrow * self.ncol)]
        
        # 定义 4 种动作导致的坐标变化量 [x轴变化, y轴变化]
        # 注意：这里的坐标系原点 (0,0) 在左上角，向右 x 增加，向下 y 增加。
        # change[0]: 上 (y减1) -> [0, -1]
        # change[1]: 下 (y加1) -> [0, 1]
        # change[2]: 左 (x减1) -> [-1, 0]
        # change[3]: 右 (x加1) -> [1, 0]
        change = [[0, -1], [0, 1], [-1, 0], [1, 0]]

        # 嵌套循环：遍历网格中的每一个位置 (i 代表行/y坐标，j 代表列/x坐标)
        for i in range(self.nrow):
            for j in range(self.ncol):
                # 遍历在当前位置 (i, j) 下可以执行的 4 种动作 (a 取值 0, 1, 2, 3)
                for a in range(4):
                    # --- 特殊情况 1：智能体当前已经处于“悬崖”或“终点” ---
                    # 如果行数 i == 3 (最后一行) 且列数 j > 0 (排除了 j=0 的起点)
                    if i == self.nrow - 1 and j > 0:
                        # P[一维状态索引][动作] = [(转移概率, 下一个状态, 奖励, 是否结束)]
                        # 一维状态索引计算法: 行号 * 总列数 + 列号 (即 i * self.ncol + j)
                        # 因为已经在终点或掉入悬崖，游戏已结束，所以无论做什么动作：
                        # 概率是 1，状态留在原地不发生变化，奖励为 0，done 为 True(结束)
                        P[i * self.ncol + j][a] = [(1, i * self.ncol + j, 0, True)]     # 以一列为一个状态，行数 * 列数 + 列数 = 状态索引

                        # 直接跳过当前动作的后续计算，进入下一个动作/状态的循环
                        continue 

                    # 其他位置
                    next_x = min(self.ncol - 1, max(0, j + change[a][0]))
                    next_y = min(self.nrow - 1, max(0, i + change[a][1]))
                    next_state = next_y * self.ncol + next_x
                    # 每一步的奖励
                    reward = -1
                    # 
                    done = False
                    # 下一个位置在悬崖或者终点（下一步的行数为最后一行且列数大于0）
                    if next_y == self.nrow - 1 and next_x > 0:
                        done = True
                        # 下一个位置在悬崖
                        if next_x != self.ncol - 1:  
                            reward = -100

                    # 完善转移矩阵 P 的内容：对于当前状态 (i, j) 和动作 a，记录转移概率、下一个状态、奖励和是否结束
                    P[i * self.ncol + j][a] = [(1, next_state, reward, done)]

        return P    

策略迭代算法的代码实现过程
$$v_\pi(s) = \sum_{a \in A} \pi(a|s) \sum_{s', r} p(s', r | s, a) [r + \gamma v_\pi(s')]$$
$$q_\pi(s, a) = \sum_{s', r} p(s', r | s, a) [r + \gamma v_\pi(s')]$$
$$\pi_{k+1} = \arg\max_\pi (r_\pi + \gamma P_\pi v_k)$$

In [ ]:
class PolicyIteration:
    """ 
        策略迭代算法 
    """
    def __init__(self, env, theta, gamma):
        """
            env：环境对象
            theta：策略评估收敛阈值
            gamma：折扣因子
            theta：策略评估收敛阈
        """
        self.env = env
        # 初始化价值为0(大小为状态数)
        self.v = [0] * self.env.ncol * self.env.nrow  
        # 初始化为均匀随机策略（每个状态选择每个动作都是一样的） 
        self.pi = [[0.25, 0.25, 0.25, 0.25]
                   for i in range(self.env.ncol * self.env.nrow)]  
        # 策略评估收敛阈，因为我们使用迭代解求解，找到最终的 0 点是比较难的；所以我们认为前后的差小于一个值即认为收敛了
        self.theta = theta
        # 折扣因子
        self.gamma = gamma

    def policy_evaluation(self):
        """
            策略评估
            因为我们并不知道当前策略下的真实的状态价值，所以我们只能通过迭代来求解状态价值
        """
        # 策略评估迭代次数
        cnt = 1

        while 1:
            # 用来记录这一轮迭代中，所有状态里价值变化的最大幅度。这是判断是否收敛的关键指标
            max_diff = 0
            # 用于存放迭代后的状态价值
            new_v = [0] * self.env.ncol * self.env.nrow

            # 对每一个状态进行迭代求解
            for s in range(self.env.ncol * self.env.nrow):
                # 开始计算状态s下的所有Q(s,a)价值
                qsa_list = [] 
                for a in range(4):
                    # 初始化当前动作 a 的动作价值为 0
                    qsa = 0
                    # 环境的状态转移模型（详细看上面的定义）
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        # 如果终止了，就和下一项无关了
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                    
                    # 这里求的是状态价值，所以需要乘以策略
                    qsa_list.append(self.pi[s][a] * qsa)

                # 状态价值函数和动作价值函数之间的关系
                new_v[s] = sum(qsa_list)  
                # 计算当前状态的新价值 new_v[s] 和旧价值 self.v[s] 之间的差值绝对值，并更新全局的最大差值 max_diff
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))
            # 更新状态价值
            self.v = new_v

            # # 满足收敛条件,退出评估迭代
            if max_diff < self.theta: 
                break

            cnt += 1
        
        print("策略评估进行%d轮后完成" % cnt)

    def policy_improvement(self):
        """
            策略提升
        """
        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = []
            for a in range(4):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
            # 上面的部分和前面策略提升部分一样，只不过并不需要进行策略迭代
            # 即我们这里只需找到最大的 Q 值即可，然后令当前状态下的动作为该 Q 值最大的动作（贪婪的做法）；或者使用软策略（soft）
            maxq = max(qsa_list)
            # 计算有几个动作得到了最大的Q值
            cntq = qsa_list.count(maxq)
            # 让这些动作均分概率（这里其实是贪婪的做法）
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]
        
        print("策略提升完成")
        return self.pi

    def policy_iteration(self):  
        """
            策略迭代(即将上面两个合并封装而已)
        """
        while 1:
            # 策略评估
            self.policy_evaluation()
            # 将列表进行深拷贝,方便接下来进行比较
            old_pi = copy.deepcopy(self.pi)
            # 策略提升
            new_pi = self.policy_improvement()
            # 如果策略收敛了就退出（即找到最优策略）
            if old_pi == new_pi: 
                break

现在我们已经写好了环境代码和策略迭代代码。为了更好地展现最终的策略，接下来增加一个打印策略的函数，用于打印当前策略在每个状态下的价值以及智能体会采取的动作。对于打印出来的动作，我们用^o<o表示等概率采取向左和向上两种动作，ooo>表示在当前状态只采取向右动作。

In [ ]:
def print_agent(agent, action_meaning, disaster=[], end=[]):
    print("状态价值：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            # 为了输出美观,保持输出6个字符
            print('%6.6s' % ('%.3f' % agent.v[i * agent.env.ncol + j]), end=' ')
        print()

    print("策略：")
    for i in range(agent.env.nrow):
        for j in range(agent.env.ncol):
            # 一些特殊的状态,例如悬崖漫步中的悬崖
            if (i * agent.env.ncol + j) in disaster:
                print('****', end=' ')
            elif (i * agent.env.ncol + j) in end:  # 目标状态
                print('EEEE', end=' ')
            else:
                a = agent.pi[i * agent.env.ncol + j]
                pi_str = ''
                for k in range(len(action_meaning)):
                    pi_str += action_meaning[k] if a[k] > 0 else 'o'
                print(pi_str, end=' ')
        print()


# 创建一个悬崖漫步环境
env = CliffWalkingEnv()
# 动作空间
action_meaning = ['^', 'v', '<', '>']
# 策略评估收敛阈
theta = 0.001
# 折扣因子
gamma = 0.9
# 创建算法实例
agent = PolicyIteration(env, theta, gamma)
# 策略迭代
agent.policy_iteration()
# 打印策略
print_agent(agent, action_meaning, list(range(37, 47)), [47])

策略评估进行60轮后完成
策略提升完成
策略评估进行72轮后完成
策略提升完成
策略评估进行44轮后完成
策略提升完成
策略评估进行12轮后完成
策略提升完成
策略评估进行1轮后完成
策略提升完成
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 


上面这个打印的策略与环境反过来了，我们是从原点开始打印的，但实际是从左下角开始出发；不影响最终结果

价值迭代算法：
<br>
价值迭代是先进行策略更新，然后再策略评估的

从上面的代码运行结果中我们能发现，策略迭代中的策略评估需要进行很多轮才能收敛得到某一策略的状态函数，这需要很大的计算量，尤其是在状态和动作空间比较大的情况下。我们是否必须要完全等到策略评估完成后再进行策略提升呢？试想一下，可能出现这样的情况：虽然状态价值函数还没有收敛，但是不论接下来怎么更新状态价值，策略提升得到的都是同一个策略。如果只在策略评估中进行一轮价值更新，然后直接根据更新后的价值进行策略提升，这样是否可以呢？答案是肯定的，这其实就是本节将要讲解的价值迭代算法，它可以被认为是一种策略评估只进行了一轮更新的策略迭代算法。需要注意的是，价值迭代中不存在显式的策略，我们只维护一个状态价值函数

下面这个和数学原理那个课中的即值迭代有点不太一样；在价值迭代中，我们需要做的一件事就是求解最大的状态价值；即求解贝尔曼公式，而求解贝尔曼公式即找到每一个状态最大的 Q 值；所以我们只需要简化为一步即可，找出当前的最大状态价值，然后找其对应的动作即可；

In [7]:
class ValueIteration:
    """ 
        价值迭代算法 
    """
    def __init__(self, env, theta, gamma):
        """
            env：环境对象
            theta：策略评估收敛阈值
            gamma：折扣因子
            theta：策略评估收敛阈
        """
        self.env = env
        # 初始化价值为0(大小为状态数)
        self.v = [0] * self.env.ncol * self.env.nrow  
        # 初始策略为0；因为我们是先进行策略迭代；即找每一个状态下的最大的 Q 值，然后直接更新；所以我们不需要一开始就有一个策略来用来评估当前策略
        self.pi = [None for i in range(self.env.ncol * self.env.nrow)] 
        # 策略评估收敛阈，因为我们使用迭代解求解，找到最终的 0 点是比较难的；所以我们认为前后的差小于一个值即认为收敛了
        self.theta = theta
        # 折扣因子
        self.gamma = gamma

    
    def value_iteration(self):
        """
            价值迭代
            先进行策略提舍，然后进行策略评估
        """
        # 迭代次数
        cnt = 0
        while 1:
            # 在每一轮迭代开始前，准备记录这一轮里所有状态中价值变化最大的那个差值。如果这个最大差值非常小，说明所有状态都算准了
            max_diff = 0
            # 新的状态价值，用于更新，同时与旧的进行比较
            new_v = [0] * self.env.ncol * self.env.nrow

            # 开始计算状态s下的所有Q(s,a)价值
            for s in range(self.env.ncol * self.env.nrow):
                qsa_list = [] 
                for a in range(4):
                    qsa = 0
                    for res in self.env.P[s][a]:
                        p, next_state, r, done = res
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))

                    qsa_list.append(qsa)
                
                # 这里不管在当前状态下的其他的 Q 值，直接选最大的作为当前的状态价值
                new_v[s] = max(qsa_list)
                # 更新重置最大差值
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))

            # 更新状态价值
            self.v = new_v

            # 满足收敛条件,退出评估迭代
            if max_diff < self.theta: 
                break  

            cnt += 1

        print("价值迭代一共进行%d轮" % cnt)

        # 根据价值函数导出一个贪婪策略
        self.get_policy()

    def get_policy(self):  
        """
            根据价值函数导出一个贪婪策略
        """
        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = []
            for a in range(4):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
            maxq = max(qsa_list)
            # 计算有几个动作得到了最大的Q值
            cntq = qsa_list.count(maxq)  
            # 让这些动作均分概率
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]


env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']
theta = 0.001
gamma = 0.9
agent = ValueIteration(env, theta, gamma)
agent.value_iteration()
print_agent(agent, action_meaning, list(range(37, 47)), [47])

价值迭代一共进行14轮
状态价值：
-7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 
-7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 
-7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000 
-7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000 
策略：
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo 
ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo 
^ooo **** **** **** **** **** **** **** **** **** **** EEEE 
